# PA 766 · Module 8 · Merging datasets

**Question:** How can we combine county information stored in separate files?

We combine internet access, social vulnerability, income, and urban/rural data using **county FIPS codes**. Each row in the final table represents one North Carolina county.

**Workflow:** load files → check county IDs → merge → check matches.


## 1 · Load the data

Run the cells in order. In **Google Colab**, the path cell connects to Drive. Locally, open this notebook in the M08 folder or change `DATA_DIR`.

The four CSVs are in the `datamerging` subfolder. We read `FIPS` as text because it is an identifier, not a quantity.


In [ ]:
from pathlib import Path  # Handles file paths.
import pandas as pd  # Works with tables.
from IPython.display import display  # Displays tables in the notebook.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/a-ncsu-courses/R-PA766-P/M08 NLP Applications/datamerging")

In [ ]:
internet = pd.read_csv(DATA_DIR / "internet.csv", dtype={"FIPS": "string"})  # Loads internet access data.
social_vulnerability = pd.read_csv(DATA_DIR / "social_vulnerability.csv", dtype={"FIPS": "string"})  # Loads social vulnerability data.
income = pd.read_csv(DATA_DIR / "mhincome.csv", dtype={"FIPS": "string"})  # Loads median household income.
urbanrural = pd.read_csv(DATA_DIR / "urbanrural.csv", dtype={"FIPS": "string"})  # Loads urban/rural labels.


## 2 · Check the matching key

A **key** identifies the same county across files. Here, `FIPS` is the key; county names and row positions are not needed for matching.

Three files contain 100 North Carolina counties. The social vulnerability file contains 3,143 counties nationwide. We will keep the counties in `internet`.


In [ ]:
for name, table in [("Internet", internet), ("Social vulnerability", social_vulnerability), ("Income", income), ("Urban/rural", urbanrural)]:  # Checks all four tables.
    print(name, ":", len(table), "rows")  # Displays the number of rows.

display(internet.head(3))  # Shows the first three counties and their internet data.
display(income.head(3))  # Shows income data with the same county IDs.


County FIPS codes have **five digits**. Some codes in the national file lost a leading zero: `1001` should be `01001`. Standardize the format and check that each file has one row per county.


In [ ]:
for table in [internet, social_vulnerability, income, urbanrural]:  # Prepares each table for merging.
    table["FIPS"] = table["FIPS"].str.strip().str.zfill(5)  # Removes spaces and adds leading zeros.
    assert table["FIPS"].notna().all(), "A FIPS code is missing."  # Checks for missing county IDs.
    assert table["FIPS"].str.fullmatch(r"[0-9]{5}").all(), "Check the FIPS format."  # Checks for five digits.
    assert table["FIPS"].is_unique, "A FIPS code appears more than once."  # Checks for duplicate county IDs.

display(social_vulnerability.head(3))  # Shows the standardized county IDs.


## 3 · Merge two datasets

A **left merge** keeps every county in the first table and adds matching information from the second.

- `on="FIPS"`: match counties by their IDs.
- `how="left"`: keep all rows from `internet`.
- `validate="one_to_one"`: require unique keys in both tables.
- `indicator=True`: add a column showing whether each county matched.


In [ ]:
ncdata = pd.merge(
    internet,  # Starts with the North Carolina counties.
    social_vulnerability,  # Adds social vulnerability information.
    on="FIPS",  # Matches rows using county IDs.
    how="left",  # Keeps every county in the first table.
    validate="one_to_one",  # Checks that each county appears once in each table.
    indicator=True,  # Records whether each county matched.
)
display(ncdata.head(3))  # Shows the combined columns.


In [ ]:
display(ncdata["_merge"].value_counts())  # Counts matched and unmatched counties.
display(ncdata.loc[ncdata["_merge"] == "left_only", ["FIPS", "NAME"]])  # Lists counties without a match.
ncdata = ncdata.drop(columns="_merge")  # Removes the match-check column after reviewing it.


`both` means a county matched. `left_only` means it appeared in `internet` without a match in the second table; the added values will be missing (`NaN`). Here, all 100 counties match.


## 4 · Add the remaining datasets

Repeat the same merge to add income and urban/rural information. The final table should still have **100 rows**, with one row per county.


In [ ]:
ncdata = pd.merge(
    ncdata,  # Uses the combined table from the previous step.
    income,  # Adds median household income.
    on="FIPS",  # Matches county IDs.
    how="left",  # Keeps all counties already in ncdata.
    validate="one_to_one",  # Checks for duplicate county IDs.
)

ncdata = pd.merge(
    ncdata,  # Uses the table that now includes income.
    urbanrural,  # Adds urban/rural information.
    on="FIPS",  # Matches county IDs.
    how="left",  # Keeps all counties already in ncdata.
    validate="one_to_one",  # Checks for duplicate county IDs.
)


In [ ]:
assert len(ncdata) == len(internet), "The number of counties changed."  # Checks the final row count.
assert ncdata["FIPS"].is_unique, "A county appears more than once."  # Checks for duplicate counties.
print("Final table:", ncdata.shape[0], "rows and", ncdata.shape[1], "columns")  # Displays the table size.
display(ncdata.isna().sum())  # Counts missing values in each column.
display(ncdata[["FIPS", "NAME", "Percent_HomesWithBroadband", "MHIncome", "UrbanRural"]].head())  # Shows selected merged columns.


## 5 · What if a county has no match?

The original files match all 100 counties. For this example, make a **practice copy** of income that omits the first county. The original `income` table stays intact.

| Merge | What it keeps |
|---|---|
| `left` | Every county from the first table, including unmatched counties |
| `inner` | Only counties found in both tables |


In [ ]:
income_demo = income.iloc[1:].copy()  # Omits one county in a practice copy.
left_demo = pd.merge(internet, income_demo, on="FIPS", how="left", validate="one_to_one")  # Keeps all internet counties.
inner_demo = pd.merge(internet, income_demo, on="FIPS", how="inner", validate="one_to_one")  # Keeps only matching counties.

print("Left merge:", len(left_demo), "rows")  # Shows that the unmatched county stays.
print("Inner merge:", len(inner_demo), "rows")  # Shows that the unmatched county is dropped.
display(left_demo.loc[left_demo["MHIncome"].isna(), ["FIPS", "NAME", "MHIncome"]])  # Shows the county with missing income.